In [0]:
import re
import csv
import io
import requests
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)

# Public GitHub raw URL for your .sql dump
raw_url = "https://raw.githubusercontent.com/jarviscanada/jarvis_data_eng_BasilSyed/refs/heads/main/python_data_analytics/psql/retail.sql"

sql_text = requests.get(raw_url, timeout=60).text
if not sql_text:
    raise ValueError("Could not download the SQL dump file.")

# Find the COPY block for public.retail
pattern = re.compile(
    r"COPY\s+public\.retail\s*\((.*?)\)\s+FROM\s+stdin;\n",
    re.IGNORECASE | re.DOTALL
)

match = pattern.search(sql_text)
if not match:
    raise ValueError("COPY block for public.retail was not found.")

columns = [c.strip() for c in match.group(1).split(",")]

start = match.end()
end = sql_text.find("\n\\.", start)
if end == -1:
    raise ValueError("Could not find end of COPY block.")

data_block = sql_text[start:end].strip()

# Parse PostgreSQL COPY data, tab-delimited
parsed_rows = []
reader = csv.reader(io.StringIO(data_block), delimiter="\t")

for row in reader:
    parsed_rows.append([
        None if v == r"\N" else v
        for v in row
    ])

# Create Spark DataFrame as strings first
schema = StructType([
    StructField("invoice_no", StringType(), True),
    StructField("stock_code", StringType(), True),
    StructField("description", StringType(), True),
    StructField("quantity_raw", StringType(), True),
    StructField("invoice_date_raw", StringType(), True),
    StructField("unit_price_raw", StringType(), True),
    StructField("customer_id_raw", StringType(), True),
    StructField("country", StringType(), True),
])

df_raw = spark.createDataFrame(parsed_rows, schema=schema)

# Cast into final types
df = (
    df_raw
    .withColumn("quantity", F.col("quantity_raw").cast("int"))
    .withColumn("invoice_date", F.to_timestamp("invoice_date_raw", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("unit_price", F.col("unit_price_raw").cast("double"))
    .withColumn("customer_id", F.col("customer_id_raw").cast("double"))
    .drop("quantity_raw", "invoice_date_raw", "unit_price_raw", "customer_id_raw")
    .select(
        "invoice_no",
        "stock_code",
        "description",
        "quantity",
        "invoice_date",
        "unit_price",
        "customer_id",
        "country"
    )
)

display(df.limit(10))

invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.95,13085.0,United Kingdom
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom
489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01T07:45:00.000Z,2.1,13085.0,United Kingdom
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom
489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01T07:45:00.000Z,1.65,13085.0,United Kingdom
489434,21871,SAVE THE PLANET MUG,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom
489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01T07:45:00.000Z,5.95,13085.0,United Kingdom
489435,22350,CAT BOWL,12,2009-12-01T07:46:00.000Z,2.55,13085.0,United Kingdom
489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01T07:46:00.000Z,3.75,13085.0,United Kingdom


In [0]:


target_table = "test_flight_data.default.retail"

(
    df.write
      .format("delta")
      .mode("overwrite")
      .saveAsTable(target_table)
)

print(f"Saved table: {target_table}")

Saved table: test_flight_data.default.retail


In [0]:
%sql

SELECT *
FROM test_flight_data.default.retail
LIMIT 20;

invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
537131,22121,NOEL WOODEN BLOCK LETTERS,2,2010-12-05T12:26:00.000Z,5.95,15716.0,United Kingdom
537131,21326,AGED GLASS SILVER T-LIGHT HOLDER,12,2010-12-05T12:26:00.000Z,0.65,15716.0,United Kingdom
537131,22809,SET OF 6 T-LIGHTS SANTA,6,2010-12-05T12:26:00.000Z,2.95,15716.0,United Kingdom
537131,22572,ROCKING HORSE GREEN CHRISTMAS,96,2010-12-05T12:26:00.000Z,0.72,15716.0,United Kingdom
537131,22576,SWALLOW WOODEN CHRISTMAS DECORATION,48,2010-12-05T12:26:00.000Z,0.85,15716.0,United Kingdom
537131,22086,PAPER CHAIN KIT 50'S CHRISTMAS,2,2010-12-05T12:26:00.000Z,2.95,15716.0,United Kingdom
537131,22910,PAPER CHAIN KIT VINTAGE CHRISTMAS,1,2010-12-05T12:26:00.000Z,2.95,15716.0,United Kingdom
537131,22365,DOORMAT RESPECTABLE HOUSE,1,2010-12-05T12:26:00.000Z,7.95,15716.0,United Kingdom
537131,22904,CALENDAR PAPER CUT DESIGN,3,2010-12-05T12:26:00.000Z,2.95,15716.0,United Kingdom
537131,22121,NOEL WOODEN BLOCK LETTERS,1,2010-12-05T12:26:00.000Z,5.95,15716.0,United Kingdom


# Retail Data Analytics Notebook

In [0]:
from pyspark.sql import functions as F

retail_df = (
    spark.table("test_flight_data.default.retail")
    .select(
        F.col("invoice_no").alias("Invoice"),
        F.col("stock_code").alias("StockCode"),
        F.col("description").alias("Description"),
        F.col("quantity").alias("Quantity"),
        F.col("invoice_date").alias("InvoiceDate"),
        F.col("unit_price").alias("Price"),
        F.col("customer_id").alias("CustomerID"),
        F.col("country").alias("Country")
    )
)

display(retail_df.limit(5))

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
537131,22121,NOEL WOODEN BLOCK LETTERS,2,2010-12-05T12:26:00.000Z,5.95,15716.0,United Kingdom
537131,21326,AGED GLASS SILVER T-LIGHT HOLDER,12,2010-12-05T12:26:00.000Z,0.65,15716.0,United Kingdom
537131,22809,SET OF 6 T-LIGHTS SANTA,6,2010-12-05T12:26:00.000Z,2.95,15716.0,United Kingdom
537131,22572,ROCKING HORSE GREEN CHRISTMAS,96,2010-12-05T12:26:00.000Z,0.72,15716.0,United Kingdom
537131,22576,SWALLOW WOODEN CHRISTMAS DECORATION,48,2010-12-05T12:26:00.000Z,0.85,15716.0,United Kingdom


## Total Invoice Amount Distribution

In [0]:
from pyspark.sql import functions as F

df_pos = retail_df.filter(
    (F.col("Quantity") > 0) & (F.col("Price") > 0)
)

inv_df = (
    df_pos
    .withColumn("LineAmount", F.col("Quantity") * F.col("Price"))
    .groupBy("Invoice")
    .agg(F.sum("LineAmount").alias("Amount"))
    .orderBy("Invoice")
)

display(inv_df.limit(5))

Invoice,Amount
489434,505.30000000000007
489435,145.79999999999998
489436,630.33
489437,310.75
489438,2286.24


## Monthly Placed and Cancelled Orders

In [0]:
from pyspark.sql import functions as F

retail_monthly_df = retail_df.withColumn(
    "InvoiceYearMonth",
    F.date_format("InvoiceDate", "yyyyMM").cast("int")
)

monthly_total_orders = (
    retail_monthly_df
    .groupBy("InvoiceYearMonth")
    .agg(F.countDistinct("Invoice").alias("Total"))
)

monthly_canceled_orders_df = (
    retail_monthly_df
    .filter(F.col("Invoice").cast("string").startswith("C"))
    .groupBy("InvoiceYearMonth")
    .agg(F.countDistinct("Invoice").alias("Cancellation"))
)

monthly_joined_df = (
    monthly_total_orders
    .join(monthly_canceled_orders_df, on="InvoiceYearMonth", how="left")
    .fillna({"Cancellation": 0})
)

monthly_with_placement_df = monthly_joined_df.withColumn(
    "Placement",
    F.col("Total") - 2 * F.col("Cancellation")
)

monthly_orders_df = (
    monthly_with_placement_df
    .select("InvoiceYearMonth", "Placement", "Cancellation")
    .orderBy("InvoiceYearMonth")
)

display(monthly_orders_df)

InvoiceYearMonth,Placement,Cancellation
200912,1528,401
201001,1033,300
201002,1489,240
201003,1553,407
201004,1284,304
201005,1604,407
201006,1502,357
201007,1329,344
201008,1331,273
201009,1633,371


## Monthly Sales

In [0]:
from pyspark.sql import functions as F

df_sales = (
    retail_df
    .filter((F.col("Quantity") > 0) & (F.col("Price") > 0))
    .withColumn("InvoiceYearMonth", F.date_format("InvoiceDate", "yyyyMM"))
    .withColumn("Sales", F.col("Quantity") * F.col("Price"))
)

monthly_sales_df = (
    df_sales
    .groupBy("InvoiceYearMonth")
    .agg(F.sum("Sales").alias("Sales"))
    .orderBy("InvoiceYearMonth")
)

display(monthly_sales_df)

InvoiceYearMonth,Sales
200912,825685.7600000115
201001,652708.5019999943
201002,553713.3060000008
201003,833570.1310000112
201004,681528.9919999797
201005,659858.8599999907
201006,752270.1399999794
201007,650712.9400000058
201008,697274.9099999849
201009,924333.0109999602


## Monthly Sales Growth

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = Window.orderBy("InvoiceYearMonth")

monthly_sales_growth_df = (
    monthly_sales_df
    .withColumn("PrevSales", F.lag("Sales").over(w))
    .withColumn(
        "GrowthPct",
        (F.col("Sales") - F.col("PrevSales")) / F.col("PrevSales")
    )
    .drop("PrevSales")
)

display(monthly_sales_growth_df)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


InvoiceYearMonth,Sales,GrowthPct
200912,825685.7600000115,null
201001,652708.5019999943,-0.2094952660925324
201002,553713.3060000008,-0.1516683108871077
201003,833570.1310000112,0.5054182768727071
201004,681528.9919999797,-0.18239753722657015
201005,659858.8599999907,-0.031796346530170334
201006,752270.1399999794,0.14004703975633523
201007,650712.9400000058,-0.13500097185829604
201008,697274.9099999849,0.07155531592775567
201009,924333.0109999602,0.32563641362053347


## Monthly Active Users

In [0]:
from pyspark.sql import functions as F

retail_df = retail_df.withColumn(
    "InvoiceYearMonth",
    F.date_format("InvoiceDate", "yyyyMM")
)

active_users_df = (
    retail_df
    .groupBy("InvoiceYearMonth")
    .agg(F.countDistinct("CustomerID").alias("ActiveUsers"))
    .orderBy("InvoiceYearMonth")
)

display(active_users_df.limit(5))

InvoiceYearMonth,ActiveUsers
200912,1045
201001,786
201002,807
201003,1111
201004,998


## New and Existing Users

In [0]:
from pyspark.sql import functions as F

df_u = (
    retail_df
    .filter(
        (F.col("Quantity") > 0) &
        (F.col("Price") > 0) &
        (F.col("CustomerID").isNotNull())
    )
    .withColumn("InvoiceYearMonth", F.date_format("InvoiceDate", "yyyyMM"))
)

first_month = (
    df_u
    .groupBy("CustomerID")
    .agg(F.min("InvoiceYearMonth").alias("FirstMonth"))
)

df_u = (
    df_u
    .join(first_month, on="CustomerID", how="left")
    .withColumn("IsNew", F.col("InvoiceYearMonth") == F.col("FirstMonth"))
)

monthly_new = (
    df_u
    .filter(F.col("IsNew"))
    .groupBy("InvoiceYearMonth")
    .agg(F.countDistinct("CustomerID").alias("NewUserCount"))
)

monthly_total = (
    df_u
    .groupBy("InvoiceYearMonth")
    .agg(F.countDistinct("CustomerID").alias("Total"))
)

users_df = (
    monthly_total
    .join(monthly_new, on="InvoiceYearMonth", how="left")
    .fillna({"NewUserCount": 0})
    .withColumn("ExistingUserCount", F.col("Total") - F.col("NewUserCount"))
    .select(
        "InvoiceYearMonth",
        F.col("NewUserCount").cast("int"),
        F.col("ExistingUserCount").cast("int")
    )
    .orderBy("InvoiceYearMonth")
)

display(users_df.limit(5))

InvoiceYearMonth,NewUserCount,ExistingUserCount
200912,955,0
201001,383,337
201002,374,398
201003,443,614
201004,294,648


## RFM Segmentation

In [0]:
from pyspark.sql import functions as F
from datetime import timedelta

df_rfm = (
    retail_df
    .filter(F.col("CustomerID").isNotNull())
    .withColumn("Sales", F.col("Quantity") * F.col("Price"))
)

max_invoice_date = df_rfm.agg(F.max("InvoiceDate").alias("MaxDate")).first()["MaxDate"]
snapshot_date = max_invoice_date + timedelta(days=1)

rfm_base_df = (
    df_rfm
    .groupBy("CustomerID")
    .agg(
        F.max("InvoiceDate").alias("LastPurchase"),
        F.countDistinct("Invoice").alias("Frequency"),
        F.sum("Sales").alias("Monetary")
    )
)

rfm_with_recency_df = rfm_base_df.withColumn(
    "Recency",
    F.datediff(F.lit(snapshot_date), F.col("LastPurchase"))
)

rfm_df = (
    rfm_with_recency_df
    .select(
        "CustomerID",
        "Recency",
        "Frequency",
        "Monetary"
    )
    .orderBy("CustomerID")
)

display(rfm_df.limit(5))

CustomerID,Recency,Frequency,Monetary
12346.0,326,17,-64.67999999999302
12347.0,3,8,5633.319999999999
12348.0,76,5,2019.4
12349.0,19,5,4404.54
12350.0,311,1,334.40000000000003


#### RFM Score Values

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

recency_window = Window.orderBy(F.col("Recency").asc())

frequency_rank_window = Window.orderBy(F.col("Frequency").asc(), F.col("CustomerID").asc())

frequency_bucket_window = Window.orderBy(F.col("FrequencyRank").asc())

monetary_window = Window.orderBy(F.col("Monetary").asc())

rfm_scored_step1 = rfm_df.withColumn(
    "RecencyBucket",
    F.ntile(5).over(recency_window)
)

rfm_scored_step2 = rfm_scored_step1.withColumn(
    "FrequencyRank",
    F.row_number().over(frequency_rank_window)
)

rfm_scored_step3 = rfm_scored_step2.withColumn(
    "FrequencyBucket",
    F.ntile(5).over(frequency_bucket_window)
)

rfm_scored_step4 = rfm_scored_step3.withColumn(
    "MonetaryBucket",
    F.ntile(5).over(monetary_window)
)

rfm_scored_df = (
    rfm_scored_step4
    .withColumn("RecencyScore", (F.lit(6) - F.col("RecencyBucket")).cast("int"))
    .withColumn("FrequencyScore", F.col("FrequencyBucket").cast("int"))
    .withColumn("MonetaryScore", F.col("MonetaryBucket").cast("int"))
)

rfm_table = (
    rfm_scored_df
    .select(
        "CustomerID",
        "Recency",
        "Frequency",
        "Monetary",
        "RecencyScore",
        "FrequencyScore",
        "MonetaryScore"
    )
    .orderBy("CustomerID")
)

display(rfm_table.limit(5))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


CustomerID,Recency,Frequency,Monetary,RecencyScore,FrequencyScore,MonetaryScore
12346.0,326,17,-64.67999999999302,2,5,1
12347.0,3,8,5633.319999999999,5,4,5
12348.0,76,5,2019.4,3,3,4
12349.0,19,5,4404.54,4,3,5
12350.0,311,1,334.40000000000003,2,1,2


#### Combined RFM Score

In [0]:
rfm_with_score_df = rfm_table.withColumn(
    "RFM_SCORE",
    F.concat(
        F.col("RecencyScore").cast("string"),
        F.col("FrequencyScore").cast("string"),
        F.col("MonetaryScore").cast("string")
    )
)

display(rfm_with_score_df.limit(5))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


CustomerID,Recency,Frequency,Monetary,RecencyScore,FrequencyScore,MonetaryScore,RFM_SCORE
12346.0,326,17,-64.67999999999302,2,5,1,251
12347.0,3,8,5633.319999999999,5,4,5,545
12348.0,76,5,2019.4,3,3,4,334
12349.0,19,5,4404.54,4,3,5,435
12350.0,311,1,334.40000000000003,2,1,2,212


#### Segmentation Map

In [0]:
rfm_with_segment_key_df = rfm_with_score_df.withColumn(
    "SegmentKey",
    F.concat(
        F.col("RecencyScore").cast("string"),
        F.col("FrequencyScore").cast("string")
    )
)

rfm_segmented_df = (
    rfm_with_segment_key_df
    .withColumn(
        "Segment",
        F.when(F.col("SegmentKey").rlike(r'^[1-2][1-2]$'), 'Hibernating')
         .when(F.col("SegmentKey").rlike(r'^[1-2][3-4]$'), 'At Risk')
         .when(F.col("SegmentKey").rlike(r'^[1-2]5$'), "Can't Lose")
         .when(F.col("SegmentKey").rlike(r'^3[1-2]$'), 'About to Sleep')
         .when(F.col("SegmentKey").rlike(r'^33$'), 'Need Attention')
         .when(F.col("SegmentKey").rlike(r'^[3-4][4-5]$'), 'Loyal Customers')
         .when(F.col("SegmentKey").rlike(r'^41$'), 'Promising')
         .when(F.col("SegmentKey").rlike(r'^51$'), 'New Customers')
         .when(F.col("SegmentKey").rlike(r'^[4-5][2-3]$'), 'Potential Loyalists')
         .when(F.col("SegmentKey").rlike(r'^5[4-5]$'), 'Champions')
         .otherwise("Other")
    )
)

rfm_final_df = (
    rfm_segmented_df
    .select(
        "CustomerID",
        "Recency",
        "Frequency",
        "Monetary",
        "RecencyScore",
        "FrequencyScore",
        "MonetaryScore",
        "RFM_SCORE",
        "Segment"
    )
    .orderBy("CustomerID")
)

display(rfm_final_df.limit(5))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


CustomerID,Recency,Frequency,Monetary,RecencyScore,FrequencyScore,MonetaryScore,RFM_SCORE,Segment
12346.0,326,17,-64.67999999999302,2,5,1,251,Can't Lose
12347.0,3,8,5633.319999999999,5,4,5,545,Champions
12348.0,76,5,2019.4,3,3,4,334,Need Attention
12349.0,19,5,4404.54,4,3,5,435,Potential Loyalists
12350.0,311,1,334.40000000000003,2,1,2,212,Hibernating


#### Segment Summary 

In [0]:
segment_summary_df = (
    rfm_final_df
    .groupBy("Segment")
    .agg(
        F.mean("Recency").alias("Recency_mean"),
        F.count("Recency").alias("Recency_count"),
        F.mean("Frequency").alias("Frequency_mean"),
        F.count("Frequency").alias("Frequency_count"),
        F.mean("Monetary").alias("Monetary_mean"),
        F.count("Monetary").alias("Monetary_count")
    )
    .orderBy("Segment")
)

display(segment_summary_df)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Segment,Recency_mean,Recency_count,Frequency_mean,Frequency_count,Monetary_mean,Monetary_count
About to Sleep,107.7012987012987,385,1.4597402597402598,385,489.91725194805196,385
At Risk,376.3549668874172,755,4.66887417218543,755,1167.566649006624,755
Can't Lose,322.3058823529412,85,17.71764705882353,85,5725.638494117646,85
Champions,8.363744075829384,844,23.73578199052133,844,10624.50508886255,844
Hibernating,466.05859375,1536,1.3365885416666667,1536,339.887240885417,1536
Loyal Customers,67.64199655765921,1162,11.91394148020654,1162,3951.1263304647186,1162
Need Attention,112.3586956521739,276,3.6666666666666665,276,1063.319532608696,276
New Customers,9.791666666666666,48,1.0,48,373.98979166666663,48
Potential Loyalists,25.31105047748977,733,2.9672578444747613,733,902.7671350613908,733
Promising,37.067796610169495,118,1.0,118,321.7648305084745,118


#### Lookups

In [0]:
rfm_555_df = (
    rfm_final_df
    .filter(F.col("RFM_SCORE") == "555")
    .orderBy("CustomerID")
)

display(rfm_555_df.limit(5))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


CustomerID,Recency,Frequency,Monetary,RecencyScore,FrequencyScore,MonetaryScore,RFM_SCORE,Segment
12359.0,8,14,8714.89,5,5,5,555,Champions
12362.0,4,14,5284.579999999999,5,5,5,555,Champions
12395.0,16,18,5046.92,5,5,5,555,Champions
12417.0,4,27,6708.210000000002,5,5,5,555,Champions
12433.0,1,11,20428.86,5,5,5,555,Champions


In [0]:
customer_18287_df = rfm_final_df.filter(F.col("CustomerID") == 18287)

display(customer_18287_df)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


CustomerID,Recency,Frequency,Monetary,RecencyScore,FrequencyScore,MonetaryScore,RFM_SCORE,Segment
18287.0,43,8,4177.889999999999,4,4,5,445,Loyal Customers
